# SBS Radar — writer cloud, preflight seguro
SK11/SK12. Modo entregado `preflight`: no autentica ni llama a Jobs, SQL, Files o inferencia. Ejecutar requiere configuración servidor cerrada, pin SHA, snapshot local entregado y contexto del Job real. No crea recursos ni activa horarios.


In [ ]:
from pathlib import Path
import os, sys, json
project_value=os.environ.get("SBS_PROJECT_ROOT", "")
project=Path(project_value).resolve() if project_value else next((p for p in [Path.cwd(),*Path.cwd().parents] if (p/"src/sbs/operations/cloud_driver.py").is_file()),None)
if project is None: raise ValueError("PROJECT_ROOT_REQUIRED")
sys.path.insert(0,str(project/"src"))
from sbs.operations.cloud_driver import preflight, load_config, execute_from_config
mode=os.environ.get("SBS_WRITER_MODE", "preflight")
if mode not in ("preflight","execute"): raise ValueError("WRITER_MODE_INVALID")
config_path=os.environ.get("SBS_WRITER_CONFIG", "")
config_pin=os.environ.get("SBS_WRITER_CONFIG_SHA256", "")
config=load_config(config_path,config_pin) if config_path and config_pin else None
print(json.dumps(preflight(config),indent=2))


In [ ]:
if mode=="execute":
    if config is None: raise ValueError("SERVER_CONFIG_AND_PIN_REQUIRED")
    # Dynamic values defined by the pinned Job task; not user authentication.
    job_id=dbutils.widgets.get("job_id")
    run_id=dbutils.widgets.get("run_id")
    release_id=dbutils.widgets.get("release_id")
    if not job_id.isdecimal() or not run_id.isdecimal(): raise ValueError("JOB_CONTEXT_REQUIRED")
    result=execute_from_config(config,project,job_id=int(job_id),run_id=int(run_id),release_id=release_id)
    print(json.dumps({k:result[k] for k in ("status","evidence_mode","cloud_acceptance","cost") if k in result},indent=2))
else:
    print(json.dumps({"status":"preflight_only","cloud_executed":False,"schedule_activated":False,"cost":None}))


## Configuración y límites
Entregar `SBS_PROJECT_ROOT`, `SBS_WRITER_CONFIG` y `SBS_WRITER_CONFIG_SHA256` desde entorno confiable del Job. Esquema/API: skills/sbs-observabilidad-operacion/references/cloud-driver.md. Dejar `SBS_WRITER_MODE=preflight` hasta ejecución autorizada. No incluir tokens, passwords o client secrets en el JSON.
No hay cluster existente observado; serverless es candidato explícito sujeto a configuración/entorno/habilitación observados. El SDK no expone entorno ni run-as históricos: se verifican settings actuales y task snapshot, con supuesto administrativo declarado.
Tabla dedicada preseed, propiedad Serializable observada y SELECT singleton exacto son requisitos; no se infiere unicidad contra administradores. Warehouse debe estar RUNNING, sin auto-start por este driver. El modo execute reutiliza seis PDFs y vectores sellados, publica preparación de 135 raw provisions, no export002141 ni runtime. Coste desconocido.

Modo de captura fijado en DriverConfig: capture_mode=sealed por defecto; remote activa únicamente seis URLs registradas y requiere capture_state_root POSIX persistente. Estado local perdido o desalineado con una publicación compartida bloquea y requiere recuperación; no asumir persistencia en serverless ni usar /Volumes como POSIX. Texto nuevo queda pending sin embeddings autorizados. Descubrimiento produce candidatos separados, no incorpora normas.
